In [8]:
from pathlib import Path
import os
from hashlib import sha256
from safetensors.torch import save_file, load_file
from fileformer.tokenizer import ByteLevelTokenizer
import torch
from torch import Tensor

In [9]:
META_END_MARKERS = [b'IDAT', b'data', b'mdat', b'\xFF\xDA', b'\x0A\x0A']
PATH_SOUSE_DATA = Path('ex')
CHUNK_SIZE = 2048
tokenizer = ByteLevelTokenizer()
Path("tt").mkdir(exist_ok=True)

In [16]:
def _tensor_from_tokens(tokens, dtype=torch.float) -> Tensor:
    return torch.tensor(tokens, dtype=dtype)

In [17]:
def process_file(file_path: Path, output_base: Path):
    """Обрабатывает один файл: отделяет метаданные, сохраняет их и данные по частям."""
    # Создаём выходную директорию для файла
    out_dir = output_base / file_path.name
    out_dir.mkdir(exist_ok=True)

    with open(file_path, 'rb') as f:
        # --- Поиск маркера конца метаданных ---
        non_empty_markers = [m for m in META_END_MARKERS if m]
        meta_buffer = bytearray()
        data_remainder = b''

        if non_empty_markers:
            search_buf = bytearray()
            max_marker_len = max(len(m) for m in non_empty_markers)
            marker_found = False

            while True:
                chunk = f.read(CHUNK_SIZE)
                if not chunk:
                    break
                search_buf.extend(chunk)

                # Поиск первого вхождения любого маркера
                best_pos = None
                best_marker = None
                for marker in non_empty_markers:
                    pos = search_buf.find(marker)
                    if pos != -1:
                        if best_pos is None or pos < best_pos:
                            best_pos = pos
                            best_marker = marker

                if best_pos is not None:
                    # Маркер найден
                    meta_buffer.extend(search_buf[:best_pos])
                    data_remainder = search_buf[best_pos + len(best_marker):]
                    marker_found = True
                    break
                else:
                    # Сохраняем часть буфера, оставляя хвост для перекрытия
                    if len(search_buf) > max_marker_len:
                        save_len = len(search_buf) - max_marker_len
                        meta_buffer.extend(search_buf[:save_len])
                        # Оставляем только хвост, где может начаться маркер
                        search_buf = search_buf[save_len:]

            if not marker_found:
                # Маркер не найден – весь файл считаем метаданными
                meta_buffer.extend(search_buf)
                data_remainder = b''
        else:
            # Нет ни одного непустого маркера – метаданных нет, весь файл — данные
            meta_buffer = bytearray()
            # Файл ещё не читался, указатель в начале

        print(f"File: {file_path.name}, metadata size: {len(meta_buffer)} bytes")

        # Сохраняем метаданные
        if meta_buffer or True:  # всегда сохраняем, даже пустые (можно убрать условие)
            hash_hex = sha256(meta_buffer).hexdigest()
            hash_tokens = _tensor_from_tokens(tokenizer.encode(hash_hex))
            meta_tokens = _tensor_from_tokens(tokenizer.encode(meta_buffer.hex()))
            save_file(
                {'hash_tokens': hash_tokens, 'tokenized_metadata': meta_tokens},
                out_dir / 'meta.safetensors'
            )

        # --- Обработка данных чанками ---
        chunk_number = 0
        current_chunk = bytearray(data_remainder)  # остаток от буфера поиска

        while True:
            # Добираем данные до полного чанка (CHUNK_SIZE)
            while len(current_chunk) < CHUNK_SIZE:
                more = f.read(CHUNK_SIZE - len(current_chunk))
                if not more:
                    break
                current_chunk.extend(more)

            if current_chunk:
                # Вычисляем хеш и токенизируем данные чанка
                hash_hex = sha256(current_chunk).hexdigest()
                hash_tokens = _tensor_from_tokens(tokenizer.encode(hash_hex))
                data_tokens = _tensor_from_tokens(tokenizer.encode(current_chunk.hex()))

                save_file(
                    {'hash_tokens': hash_tokens, 'tokenized_data': data_tokens},
                    out_dir / f'data{chunk_number}.safetensors'
                )

                chunk_number += 1
                current_chunk = bytearray()  # готовим для следующего чанка
            else:
                break

        print(f"Total chunks: {chunk_number}\n")

In [18]:
output_root = Path("tt")
output_root.mkdir(exist_ok=True)

for file in PATH_SOUSE_DATA.iterdir():
    if not file.is_file():
        continue
    process_file(file, output_root)

File: tt.png, metadata size: 87 bytes
Total chunks: 355

